# CatBoost
Accompanies the notes on **CatBoost**.

Needs the `catboost` package (`pip install catboost`).

## 1. Ordered target statistics by hand

In [ ]:
import numpy as np
import pandas as pd

def ordered_ts(categories, y, order, a=1.0):
    """Ordered target statistic: each row is encoded using only the rows before it in `order`."""
    prior = np.mean(y)
    sums, counts = {}, {}
    encoded = np.empty(len(y))
    for i in order:                                  # walk through the rows in permutation order
        c = categories[i]
        encoded[i] = (sums.get(c, 0.0) + a * prior) / (counts.get(c, 0) + a)
        sums[c] = sums.get(c, 0.0) + y[i]            # only now does row i's label become "past"
        counts[c] = counts.get(c, 0) + 1
    return encoded

device = np.array(["Desktop", "Mobile", "Desktop", "Tablet", "Mobile", "Desktop", "Mobile"])
approved = np.array([1, 0, 0, 1, 1, 1, 0])
order = [1, 3, 0, 5, 2, 4, 6]                        # rows 2, 4, 1, 6, 3, 5, 7 (counting from 1)
print(pd.DataFrame({"device": device, "y": approved,
                    "ordered TS": ordered_ts(device, approved, order).round(4)}, index=range(1, 8)))

## 2. Naive target encoding vs ordered statistics on pure noise

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n = 10000
noise_id = rng.integers(0, 3000, n).astype(str)      # 3000 "customer segments", unrelated to the target
y_noise = rng.binomial(1, 0.3, n)
id_train, id_test, yn_train, yn_test = train_test_split(noise_id, y_noise, test_size=0.3, random_state=0)

# naive target encoding: category mean computed on the training rows, each row's own label included
means = pd.Series(yn_train).groupby(id_train).mean()
naive_train = pd.Series(id_train).map(means).values
naive_test = pd.Series(id_test).map(means).fillna(yn_train.mean()).values
# ordered target statistics on the training rows; test rows use all training rows
ots_train = ordered_ts(id_train, yn_train, rng.permutation(len(yn_train)))
sums = pd.Series(yn_train).groupby(id_train).sum(); counts = pd.Series(yn_train).groupby(id_train).count()
ots_test = ((pd.Series(id_test).map(sums).fillna(0) + yn_train.mean()) / (pd.Series(id_test).map(counts).fillna(0) + 1)).values

for name, tr, te in [("naive target encoding", naive_train, naive_test), ("ordered target statistic", ots_train, ots_test)]:
    tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(tr.reshape(-1, 1), yn_train)
    print(f"{name:25s}: train AUC {roc_auc_score(yn_train, tree.predict_proba(tr.reshape(-1, 1))[:, 1]):.3f}"
          f"   test AUC {roc_auc_score(yn_test, tree.predict_proba(te.reshape(-1, 1))[:, 1]):.3f}")

## 3. Bank data with raw categorical columns

In [ ]:
import time
from catboost import CatBoostClassifier

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
cat_cols = bd.select_dtypes(include="object").columns.tolist()
print("categorical columns:", {c: bd[c].nunique() for c in cat_cols})
x_train, x_test, y_train, y_test = train_test_split(bd, y, test_size=0.25, random_state=1, stratify=y)
x_fit, x_val, y_fit, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

start = time.time()
cb = CatBoostClassifier(iterations=3000, learning_rate=0.05, eval_metric="Logloss", random_seed=0, verbose=0,
                        allow_writing_files=False)
cb.fit(x_fit, y_fit, cat_features=cat_cols, eval_set=(x_val, y_val), early_stopping_rounds=100)
print(f"trees: {cb.get_best_iteration() + 1}, time {time.time() - start:.0f}s")
print("test AUC (raw categories):", round(roc_auc_score(y_test, cb.predict_proba(x_test)[:, 1]), 4))

## 4. Feature importance

In [ ]:
imp = pd.Series(cb.get_feature_importance(), index=x_fit.columns).sort_values(ascending=False)
print(imp.round(1).head(8))

## 5. Symmetric vs depth-wise trees

In [ ]:
for policy in ["SymmetricTree", "Depthwise"]:
    m = CatBoostClassifier(iterations=3000, learning_rate=0.05, grow_policy=policy, random_seed=0, verbose=0,
                           allow_writing_files=False)
    m.fit(x_fit, y_fit, cat_features=cat_cols, eval_set=(x_val, y_val), early_stopping_rounds=100)
    print(f"{policy:13s}: trees {m.get_best_iteration() + 1:4d}, validation log-loss {m.get_best_score()['validation']['Logloss']:.4f},"
          f" test AUC {roc_auc_score(y_test, m.predict_proba(x_test)[:, 1]):.4f}")

## 6. Loan data: do the categorical columns help?

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
num = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
       "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
cats = ["State", "Loan.Purpose", "Home.Ownership", "Employment.Length"]
for col in num:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld[cats] = ld[cats].fillna("missing").astype(str)
ld = ld.dropna(subset=["Interest.Rate"])
print("categories per column:", ld[cats].nunique().to_dict())
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[num + cats], ld["Interest.Rate"], test_size=0.2, random_state=2)
xr_fit, xr_val, yr_fit, yr_val = train_test_split(xr_train, yr_train, test_size=0.2, random_state=0)
for features in [num, num + cats]:
    reg = CatBoostRegressor(iterations=3000, learning_rate=0.05, random_seed=0, verbose=0,
                            allow_writing_files=False)
    reg.fit(xr_fit[features], yr_fit, cat_features=[c for c in features if c in cats],
            eval_set=(xr_val[features], yr_val), early_stopping_rounds=100)
    print(f"{'numeric only' if features == num else 'with categories':16s}: trees {reg.get_best_iteration() + 1}, "
          f"test MAE {mean_absolute_error(yr_test, reg.predict(xr_test[features])):.3f}")

## Try it yourself
1. Reverse the order in section 1 (`order[::-1]`).
2. Bank data with `boosting_type="Plain"`.